# KM V8 events and AFT covariates — single raw pull

For each market-day this workflow pulls the preceding, target, and following raw trade days once; constructs one event-time stream; detects regular-grid 1s Lee–Mykland shocks for both spot-first and perp-first perspectives; applies the saved event-store V5 resolution rules; and attaches leakage-safe base covariates from the same in-memory data. Daily checkpoints make the run resumable and memory is released after every market-day.

In [1]:
from pathlib import Path
import gc
import os
import pandas as pd

from km_v8_regular_lm_pilot import (
    MARKETS, METHODOLOGY_VERSION, atomic_csv,
    plot_pooled_km_comparison, run_date_block,
)
from survival_analysis_data_processing_final import augment_liquidity_metrics
from survival_analysis_data_pull_final import pull_or_load_market_metrics
from survival_analysis_utils_final import (
    COVARIATE_TIMING_VERSION, validate_augmented_timing,
    validate_base_covariate_timing,
)

In [2]:
# Edit only this cell for the final run. END_DATE is exclusive.
START_DATE = os.getenv('KMV8_START_DATE', '2025-07-01')
END_DATE = os.getenv('KMV8_END_DATE', '2025-08-01')
MARKET_NAMES = os.getenv('KMV8_MARKETS', 'btc_um,btc_cm,eth_um,eth_cm').split(',')
GRID = '1s'
SIGNIFICANCE = 0.001
OUTPUT_ROOT = Path(os.getenv('KMV8_OUTPUT_ROOT', 'sa_results/km_v8_final_single_pull'))
BATCH_DAYS = 17
RUN_LIQUIDITY_AUGMENTATION = os.getenv('KMV8_LIQUIDITY', '1') == '1'
REDOWNLOAD_METRICS = False

DATES = pd.date_range(START_DATE, END_DATE, inclusive='left', freq='D').strftime('%Y-%m-%d').tolist()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(METHODOLOGY_VERSION, COVARIATE_TIMING_VERSION, len(DATES), 'days', 'batch=', BATCH_DAYS)

v8_regular_lm_daily_v4 prior_day_completed_ffill_1s_strict_pre_shock_5min_final_v3 1 days batch= 17


In [3]:
# One raw pull per 17 target days; LM/event logic remains independent by UTC day.
summary_path = OUTPUT_ROOT / 'comparison_summary.csv'
summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

def day_complete(market, date):
    if summary.empty or not {'methodology_version', 'aft_file'}.issubset(summary.columns):
        return False
    prior = summary[
        summary['market'].eq(market)
        & summary['date'].astype(str).eq(date)
        & summary['grid'].eq(GRID)
        & summary['methodology_version'].eq(METHODOLOGY_VERSION)
    ]
    return (prior['first'].nunique() == 2
            and prior['event_file'].map(lambda p: Path(p).exists()).all()
            and prior['aft_file'].map(lambda p: Path(p).exists()).all())

for market in MARKET_NAMES:
    for offset in range(0, len(DATES), BATCH_DAYS):
        batch = DATES[offset:offset + BATCH_DAYS]
        if all(day_complete(market, date) for date in batch):
            print('[skip verified block]', market, batch[0], batch[-1])
            continue
        rows = run_date_block(
            market=market, dates=batch, grids=[GRID],
            output_root=OUTPUT_ROOT, significance=SIGNIFICANCE,
            build_covariates=True,
        )
        summary = pd.concat([summary, pd.DataFrame(rows)], ignore_index=True)
        summary = summary.drop_duplicates(['market', 'date', 'first', 'grid'], keep='last')
        atomic_csv(summary.sort_values(['market', 'date', 'grid', 'first']), summary_path)
        gc.collect()

print('completed rows:', len(summary))

[skip verified block] btc_cm 2025-07-06 2025-07-06
completed rows: 2


In [4]:
# Pool daily base-covariate checkpoints into monthly files used downstream.
monthly_root = OUTPUT_ROOT / 'aft_data_monthly'
for market in MARKET_NAMES:
    for first in ('spot', 'perp'):
        paths = sorted((OUTPUT_ROOT / 'aft_data' / market).glob(f'{first}_*_{GRID}.parquet'))
        frames = [pd.read_parquet(path) for path in paths]
        if not frames:
            continue
        pooled = pd.concat(frames, ignore_index=True).drop_duplicates(['start_ts', 'resolution_pct'])
        validate_base_covariate_timing(pooled, f'{market} {first} pooled')
        for period, month in pooled.groupby(pd.to_datetime(pooled['start_ts']).dt.to_period('M')):
            out = monthly_root / market / f'{first}_{period}.parquet'
            out.parent.mkdir(parents=True, exist_ok=True)
            month.sort_values('start_ts').to_parquet(out, index=False, compression='zstd')
        del frames, pooled
        gc.collect()

In [5]:
# Optional: attach open interest and long/short metrics strictly before each event.
if RUN_LIQUIDITY_AUGMENTATION:
    for market in MARKET_NAMES:
        symbol, cm_um, _ = MARKETS[market]
        inputs = sorted((monthly_root / market).glob('*.parquet'))
        metrics = pull_or_load_market_metrics(
            OUTPUT_ROOT / 'metric_runs' / market, symbol, cm_um, inputs,
            redownload=REDOWNLOAD_METRICS,
        )
        for path in inputs:
            out = OUTPUT_ROOT / 'aft_data_liquidity_monthly' / market / path.name
            out.parent.mkdir(parents=True, exist_ok=True)
            augmented = augment_liquidity_metrics(pd.read_parquet(path), metrics)
            validate_augmented_timing(augmented, path.name)
            augmented.to_parquet(out, index=False, compression='zstd')
            del augmented
        del metrics
        gc.collect()

In [6]:
# KM comparison graph and final checkpoint audit.
summary = pd.read_csv(summary_path)
plot_pooled_km_comparison(summary, OUTPUT_ROOT)
expected = len(MARKET_NAMES) * len(DATES) * 2
current = summary[summary['methodology_version'].eq(METHODOLOGY_VERSION)]
assert len(current) == expected, (len(current), expected)
assert current['event_file'].map(lambda p: Path(p).exists()).all()
assert current['aft_file'].map(lambda p: Path(p).exists()).all()
print('verified direction-days:', len(current))
print('KM plot:', OUTPUT_ROOT / 'plots' / f'km_curves_{GRID}_v5_vs_v8.png')

verified direction-days: 2
KM plot: sa_results\km_v8_notebook_smoke\plots\km_curves_1s_v5_vs_v8.png
